In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go



df_eda = pd.read_csv("data/bronce/df_bronce.csv")

# Limpiar cadenas de texto que representan nulos
cols_texto = df_eda.select_dtypes(
    include=["object", "string"]
).columns

for col in cols_texto:
    df_eda[col] = (
        df_eda[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "NULL": pd.NA,
            "NaN": pd.NA
        })
    )

# Fechas
columnas_fecha = [
    "FECHA_DE_CREACION_OP",
    "FECHA_ULTIMA_ACTUALIZACION_OP",
    "FECHA_INICIO_VENTAS_TRANSACCIONAL"
]

for col in columnas_fecha:
    if col in df_eda.columns:
        df_eda[col] = pd.to_datetime(
            df_eda[col],
            errors="coerce",
            format="mixed",
            dayfirst=False
        )

# Vendedor como identificador categórico
if "BK_VENDEDOR" in df_eda.columns:
    df_eda["BK_VENDEDOR"] = (
        df_eda["BK_VENDEDOR"].astype("string")
    )

# Validar target
print(df_eda["COMPRA"].value_counts(dropna=False))
print("Dimensiones:", df_eda.shape)

COMPRA
1    1448
0    1335
Name: count, dtype: int64
Dimensiones: (2783, 19)


In [2]:
perfil = pd.DataFrame({
    "Variable": df_eda.columns,
    "Tipo": df_eda.dtypes.astype(str).values,
    "Nulos": df_eda.isna().sum().values,
    "Nulos_pct": (
        df_eda.isna().mean() * 100
    ).round(2).values,
    "Valores_unicos": df_eda.nunique().values
})

perfil = perfil.sort_values(
    "Nulos_pct",
    ascending=False
)

display(perfil)

,Variable,Tipo,Nulos,Nulos_pct,Valores_unicos
13,FECHA_INICIO_VENTAS_TRANSACCIONAL,"datetime64[us, UTC]",1465,52.64,44
0,Unnamed: 0,int64,0,0.00,2783
10,TIPO_VENTA,string,0,0.00,5
17,CIUDAD_COMPRADOR1,string,0,0.00,188
16,OCUPACION_COMPRADOR1,string,0,0.00,10
15,OBSERVACIONES,string,0,0.00,692
14,BK_VENDEDOR,string,0,0.00,189
12,ESTADO_PROYECTO,string,0,0.00,1
11,SALA_DE_VENTAS_ALTERNATIVA,string,0,0.00,34
9,DESTINO,string,0,0.00,4


In [3]:
fig = px.bar(
    perfil.sort_values("Nulos_pct"),
    x="Nulos_pct",
    y="Variable",
    orientation="h",
    text="Nulos_pct",
    title="Porcentaje de valores faltantes por variable",
    color="Nulos_pct",
    color_continuous_scale="Reds",
    height=700
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Porcentaje de valores faltantes",
    yaxis_title="",
    coloraxis_showscale=False
)

fig.show()

In [4]:
resultados_nulos = []

for col in df_eda.columns.drop("COMPRA"):

    temporal = df_eda[[col, "COMPRA"]].copy()

    temporal["Estado"] = np.where(
        temporal[col].isna(),
        "Faltante",
        "Disponible"
    )

    tasas = temporal.groupby("Estado")["COMPRA"].agg(
        ["mean", "count"]
    )

    for estado, fila in tasas.iterrows():
        resultados_nulos.append({
            "Variable": col,
            "Estado": estado,
            "Tasa_compra": fila["mean"] * 100,
            "N": int(fila["count"])
        })

df_nulos_compra = pd.DataFrame(resultados_nulos)

fig = px.bar(
    df_nulos_compra,
    x="Variable",
    y="Tasa_compra",
    color="Estado",
    barmode="group",
    hover_data=["N"],
    title="Tasa de compra según disponibilidad del dato"
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    yaxis_title="Tasa de compra (%)",
    height=600
)

fig.show()

In [5]:
categoricas = [
    "ETAPA_DE_VENTA",
    "REFERIDO",
    "TIPO_DE_PROYECTO",
    "METODO_DE_VENTAS",
    "CIUDAD",
    "DESTINO",
    "TIPO_VENTA",
    "SALA_DE_VENTAS_ALTERNATIVA",
    "BK_VENDEDOR",
    "OCUPACION_COMPRADOR1",
    "CIUDAD_COMPRADOR1"
]

In [6]:
def graficar_distribucion(df, variable, top_n=15):

    datos = (
        df[variable]
        .fillna("SIN INFORMACIÓN")
        .value_counts()
        .head(top_n)
        .reset_index()
    )

    datos.columns = ["Categoria", "Frecuencia"]

    fig = px.bar(
        datos.sort_values("Frecuencia"),
        x="Frecuencia",
        y="Categoria",
        orientation="h",
        text="Frecuencia",
        title=f"Distribución de {variable}",
        color="Frecuencia",
        color_continuous_scale="Blues",
        height=max(400, len(datos) * 32)
    )

    fig.update_layout(
        template="plotly_white",
        showlegend=False,
        coloraxis_showscale=False,
        yaxis_title="",
        xaxis_title="Número de oportunidades"
    )

    fig.show()

In [7]:
for variable in categoricas:
    if variable in df_eda.columns:
        graficar_distribucion(
            df_eda,
            variable,
            top_n=15
        )

####

In [9]:
def conversion_por_categoria(
    df,
    variable,
    top_n=15,
    min_n=20
):

    datos = df[[variable, "COMPRA"]].copy()

    datos[variable] = (
        datos[variable]
        .fillna("SIN INFORMACIÓN")
        .astype(str)
    )

    resumen = (
        datos.groupby(variable)
        .agg(
            Oportunidades=("COMPRA", "size"),
            Compras=("COMPRA", "sum"),
            Tasa_compra=("COMPRA", "mean")
        )
        .reset_index()
    )

    resumen["Tasa_compra"] *= 100

    # Evitar tasas extremas de muestras diminutas
    resumen = resumen[
        resumen["Oportunidades"] >= min_n
    ]

    resumen = (
        resumen
        .sort_values("Oportunidades", ascending=False)
        .head(top_n)
        .sort_values("Tasa_compra")
    )

    fig = px.bar(
        resumen,
        x="Tasa_compra",
        y=variable,
        orientation="h",
        color="Tasa_compra",
        color_continuous_scale="Teal",
        text="Tasa_compra",
        custom_data=["Oportunidades", "Compras"],
        title=f"Tasa de compra por {variable}",
        height=max(400, len(resumen) * 38)
    )

    fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        hovertemplate=(
            "%{y}<br>"
            "Conversión: %{x:.1f}%<br>"
            "Oportunidades: %{customdata[0]}<br>"
            "Compras: %{customdata[1]}"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Tasa de compra (%)",
        xaxis_range=[0, 105],
        yaxis_title="",
        coloraxis_showscale=False
    )

    fig.show()

    return resumen

In [10]:
variables_conversion = [
    "REFERIDO",
    "TIPO_DE_PROYECTO",
    "METODO_DE_VENTAS",
    "CIUDAD",
    "TIPO_VENTA",
    "DESTINO",
    "SALA_DE_VENTAS_ALTERNATIVA",
    "OCUPACION_COMPRADOR1"
]

for variable in variables_conversion:
    if variable in df_eda.columns:
        conversion_por_categoria(
            df_eda,
            variable,
            top_n=15,
            min_n=20
        )

In [12]:
variable = "REFERIDO"

resumen = (
    df_eda.groupby(variable, dropna=False)
    .agg(
        Oportunidades=("COMPRA", "size"),
        Compras=("COMPRA", "sum"),
        Tasa_compra=("COMPRA", "mean")
    )
    .reset_index()
)

resumen["Tasa_compra"] *= 100

fig = px.scatter(
    resumen,
    x="Oportunidades",
    y="Tasa_compra",
    size="Oportunidades",
    color="Tasa_compra",
    hover_name=variable,
    hover_data=["Compras"],
    title="Volumen de oportunidades vs. tasa de compra",
    color_continuous_scale="Viridis",
    size_max=45
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Número de oportunidades",
    yaxis_title="Tasa de compra (%)"
)

fig.show()

####

In [13]:
df_temporal = df_eda.dropna(
    subset=["FECHA_DE_CREACION_OP"]
).copy()

df_temporal["MES_CREACION"] = (
    df_temporal["FECHA_DE_CREACION_OP"]
    .dt.to_period("M")
    .astype(str)
)

evolucion = (
    df_temporal.groupby("MES_CREACION")
    .agg(
        Oportunidades=("COMPRA", "size"),
        Compras=("COMPRA", "sum"),
        Tasa_compra=("COMPRA", "mean")
    )
    .reset_index()
)

evolucion["Tasa_compra"] *= 100

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=evolucion["MES_CREACION"],
        y=evolucion["Oportunidades"],
        name="Oportunidades"
    )
)

fig.add_trace(
    go.Scatter(
        x=evolucion["MES_CREACION"],
        y=evolucion["Tasa_compra"],
        name="Tasa de compra",
        mode="lines+markers",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Evolución mensual de oportunidades y conversión observada",
    template="plotly_white",
    yaxis=dict(title="Oportunidades"),
    yaxis2=dict(
        title="Tasa de compra (%)",
        overlaying="y",
        side="right",
        range=[0, 100]
    ),
    hovermode="x unified"
)

fig.show()

/var/folders/_f/nq9r93ln5q97z8qph75wm8gc0000gn/T/ipykernel_11222/3881081777.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


In [23]:


# 1. Convertir fechas a datetime con zona horaria
df_eda["FECHA_DE_CREACION_OP"] = (
    pd.to_datetime(
        df_eda["FECHA_DE_CREACION_OP"],
        errors="coerce",
        utc=True
    ).dt.tz_convert("America/Bogota")
)

# 2. Definir fecha de corte
fecha_corte = pd.Timestamp(
    "2026-11-03 23:59:59",
    tz="America/Bogota"
)

# 3. Calcular antigüedad en días
df_eda["ANTIGUEDAD_OP_DIAS"] = (
    fecha_corte - df_eda["FECHA_DE_CREACION_OP"]
).dt.total_seconds() / 86400

# 4. Eliminar antigüedades negativas
df_eda.loc[
    df_eda["ANTIGUEDAD_OP_DIAS"] < 0,
    "ANTIGUEDAD_OP_DIAS"
] = np.nan

# 5. Preparar datos para visualización
df_grafico = df_eda[
    ["COMPRA", "ANTIGUEDAD_OP_DIAS"]
].dropna().copy()

df_grafico["Resultado"] = df_grafico["COMPRA"].map({
    0: "No compra",
    1: "Compra"
})

# 6. Crear histograma
fig = px.histogram(
    df_grafico,
    x="ANTIGUEDAD_OP_DIAS",
    color="Resultado",
    nbins=30,
    barmode="overlay",
    marginal="box",
    opacity=0.65,
    title="Distribución de antigüedad de oportunidades según compra",
    labels={
        "ANTIGUEDAD_OP_DIAS": "Antigüedad de la oportunidad (días)",
        "Resultado": "Resultado comercial"
    },
    color_discrete_map={
        "No compra": "#EF553B",
        "Compra": "#00CC96"
    }
)

# 7. Personalización
fig.update_layout(
    template="plotly_white",
    width=950,
    height=550,
    xaxis_title="Antigüedad de la oportunidad (días)",
    yaxis_title="Número de oportunidades",
    legend_title="Resultado comercial",
    bargap=0.05,
    hovermode="x unified"
)

fig.show()

#### 

In [15]:
df_valor = df_eda[
    ["IMPORTE_TOTAL_OP_PRIMERA_VERSION", "COMPRA"]
].copy()

df_valor["VALOR_MILLONES"] = (
    df_valor["IMPORTE_TOTAL_OP_PRIMERA_VERSION"] / 1_000_000
)

df_valor["Resultado"] = df_valor["COMPRA"].map({
    0: "No compra",
    1: "Compra"
})

fig = px.box(
    df_valor,
    x="Resultado",
    y="VALOR_MILLONES",
    color="Resultado",
    points="outliers",
    title="Distribución del valor inicial de oportunidad por resultado",
    labels={
        "VALOR_MILLONES": "Valor inicial (millones COP)",
        "Resultado": "Resultado comercial"
    }
)

fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

In [16]:
df_valor = df_valor.dropna(
    subset=["VALOR_MILLONES", "COMPRA"]
).copy()

df_valor["RANGO_VALOR"] = pd.qcut(
    df_valor["VALOR_MILLONES"],
    q=5,
    duplicates="drop"
)

resumen_valor = (
    df_valor.groupby(
        "RANGO_VALOR",
        observed=True
    )
    .agg(
        Oportunidades=("COMPRA", "size"),
        Tasa_compra=("COMPRA", "mean")
    )
    .reset_index()
)

resumen_valor["Tasa_compra"] *= 100
resumen_valor["RANGO_VALOR"] = (
    resumen_valor["RANGO_VALOR"].astype(str)
)

fig = px.bar(
    resumen_valor,
    x="RANGO_VALOR",
    y="Tasa_compra",
    text="Tasa_compra",
    hover_data=["Oportunidades"],
    title="Tasa de compra por quintiles de valor de oportunidad"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Rango de valor (millones COP)",
    yaxis_title="Tasa de compra (%)",
    yaxis_range=[0, 105]
)

fig.show()

####

In [17]:
tabla_etapa = pd.crosstab(
    df_eda["ETAPA_DE_VENTA"],
    df_eda["COMPRA"],
    normalize="index"
) * 100

display(tabla_etapa.round(2))

fig = px.imshow(
    tabla_etapa,
    text_auto=".1f",
    color_continuous_scale="Blues",
    title="Porcentaje de compra por etapa de venta",
    labels={
        "x": "Compra",
        "y": "Etapa de venta",
        "color": "Porcentaje"
    },
    aspect="auto"
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

COMPRA,0,1
ETAPA_DE_VENTA,,
0. Transición,100.00,0.00
1. Visita,98.36,1.64
2. Separación,0.00,100.00
3. Promesa,0.00,100.00
4. Escrituración,0.00,100.00
5. Entrega,0.00,100.00


In [22]:

# Convertir fechas y normalizar a UTC
df_eda["FECHA_ULTIMA_ACTUALIZACION_OP"] = pd.to_datetime(
    df_eda["FECHA_ULTIMA_ACTUALIZACION_OP"],
    errors="coerce",
    utc=True
)

# Fecha de corte con zona horaria UTC
fecha_corte = pd.Timestamp("2026-11-03", tz="UTC")

# Calcular días desde última actualización
df_eda["DIAS_ULTIMA_ACTUALIZACION"] = (
    fecha_corte - df_eda["FECHA_ULTIMA_ACTUALIZACION_OP"]
).dt.total_seconds() / 86400

# Preparar datos
df_grafico = df_eda[
    ["COMPRA", "DIAS_ULTIMA_ACTUALIZACION"]
].dropna().copy()

df_grafico["Resultado"] = df_grafico["COMPRA"].map({
    0: "No compra",
    1: "Compra"
})

# Boxplot
fig = px.box(
    df_grafico,
    x="Resultado",
    y="DIAS_ULTIMA_ACTUALIZACION",
    color="Resultado",
    points="outliers",
    title="Recencia de actualización según resultado comercial",
    labels={
        "Resultado": "Resultado comercial",
        "DIAS_ULTIMA_ACTUALIZACION": "Días desde última actualización"
    },
    color_discrete_map={
        "No compra": "#EF553B",
        "Compra": "#00CC96"
    }
)

fig.update_layout(
    template="plotly_white",
    width=850,
    height=500,
    showlegend=False
)

fig.show()


#8.2. Recencia de actualización y compra

###

In [20]:
resumen_cruzado = (
    df_eda.groupby(
        ["CIUDAD", "TIPO_DE_PROYECTO"],
        dropna=False
    )
    .agg(
        N=("COMPRA", "size"),
        Tasa_compra=("COMPRA", "mean")
    )
    .reset_index()
)

# Exigir al menos 10 oportunidades por combinación
resumen_cruzado = resumen_cruzado[
    resumen_cruzado["N"] >= 10
].copy()

resumen_cruzado["Tasa_compra"] *= 100

matriz = resumen_cruzado.pivot(
    index="CIUDAD",
    columns="TIPO_DE_PROYECTO",
    values="Tasa_compra"
)

fig = px.imshow(
    matriz,
    text_auto=".1f",
    color_continuous_scale="YlGnBu",
    zmin=0,
    zmax=100,
    title="Tasa de compra por ciudad y tipo de proyecto",
    aspect="auto",
    labels={"color": "Compra (%)"}
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()